# Potential and Limitation of QELMs

In this notebook I want to reprodue the results of https://arxiv.org/abs/2210.00780.

First of all I want to generate a training dataset, a set of denisty matrices with the corresponding expectation value of chosen observable.

In [20]:
import numpy as np
import qutip as qt

#let's define the Pauli matrices
I = np.array([[1, 0],
              [0, 1]], dtype=complex)

X = np.array([[0, 1],
              [1, 0]], dtype=complex)

Y = np.array([[0, -1j],
              [1j,  0]], dtype=complex)

Z = np.array([[1,  0],
              [0, -1]], dtype=complex)

Let's generate the training and test datasets

In [22]:
N_tr = 100
N_test = 1000

#density matrices
rho_training = np.zeros((N_tr, 2, 2), dtype=complex)
rho_test = np.zeros((N_test,2,2), dtype=complex)

#expectation values
expe_X_train = np.zeros(N_tr)
expe_X_test = np.zeros(N_test)

In [24]:
for i in range(N_tr):
    # Sample a random Bloch vector uniformly in the Bloch ball
    r = np.random.normal(size=3)
    r = r / np.linalg.norm(r) * np.random.rand()**(1/3)
    
    # Construct density matrix
    rho_training[i] = 0.5 * (I + r[0]*X + r[1]*Y + r[2]*Z)

    # Expectation value of Pauli X
    expe_X_train[i] = np.real(np.trace(rho_training[i] @ X))

for i in range(N_test):
    # Sample a random Bloch vector uniformly in the Bloch ball
    r = np.random.normal(size=3)
    r = r / np.linalg.norm(r) * np.random.rand()**(1/3)
    
    # Construct density matrix
    rho_test[i] = 0.5 * (I + r[0]*X + r[1]*Y + r[2]*Z)

    # Expectation value of Pauli X
    expe_X_test[i] = np.real(np.trace(rho_test[i] @ X))

Simulating the evoultion, let's generate the random isometry

In [70]:
dim_in = 2
dim_out = [2, 5]
total_dim_out = np.prod(dim_out) # 10
dim_env = 5

U = qt.rand_unitary(d_out)

# Extract the first 2 columns to form the isometry
# We convert to a dense array to slice easily, then back to Qobj
V_matrix = U.full()[:, :dim_in]

# Wrap it in a Qobj and set the dimensions correctly
V = qt.Qobj(V_matrix, dims=[dim_out, [dim_in]])

# Check if V.dag() * V results in a 2x2 Identity matrix
identity_check = V.dag() * V
print(identity_check)

rho_target = qt.Qobj(rho_training[5], dims=[[2], [2]])

A_out = V * rho_target * V.dag()
B_out = A_out.ptrace(1)

print(A_out.dims)
print(B_out)
print(B_out.dims)
print((V * rho_target * V.dag()).ptrace(1))

# Create a list of projectors [|0><0|, |1><1|, ..., |4><4|]
povm_elements_5 = [qt.basis(dim_env, i).proj() for i in range(dim_env)]

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
Qobj data =
[[1.00000000e+00 4.85722573e-17]
 [4.85722573e-17 1.00000000e+00]]
[[2, 5], [2, 5]]
Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.14164824-9.65433369e-18j  0.03890563-9.22871850e-02j
   0.11311144+9.16831283e-02j  0.06483438+1.20733474e-01j
   0.03844162+5.76243206e-02j]
 [ 0.03890563+9.22871850e-02j  0.11493449+2.09872799e-19j
  -0.07776709+1.05388430e-01j -0.03941651+1.03004629e-01j
   0.0194268 +7.64949070e-02j]
 [ 0.11311144-9.16831283e-02j -0.07776709-1.05388430e-01j
   0.29627039-3.33520450e-18j  0.20250617-1.15517742e-02j
  -0.01427674-5.68086919e-02j]
 [ 0.06483438-1.20733474e-01j -0.03941651-1.03004629e-01j
   0.20250617+1.15517742e-02j  0.28014015-3.06760606e-18j
   0.07058108-4.83332964e-02j]
 [ 0.03844162-5.76243206e-02j  0.0194268 -7.64949070e-02j
  -0.01427674+5.68086919e-02j  0.07058108+4.83332964e-02j
   0.16700672+1.52

In [76]:
print(povm_elements_5)

# Assuming rho_env is your result from ptrace(1)
probabilities = [qt.expect(E, B_out) for E in povm_elements_5]

# Print the distribution
for i, p in enumerate(probabilities):
    print(f"Outcome {i}: {p:.4f}")

# Verification: sum of probabilities should be 1.0
print(f"Total probability: {sum(probabilities)}")

print(probabilities)

probabilities_2 = qt.expect(povm_elements_5, B_out)

print("--------------------------------------")
print(probabilities_2)

[Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=CSR, isherm=True
Qobj data =
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]], Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]], Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]], Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0.]], Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]]]
Outcome 0: 0.1416
Outcome 1: 0.1149
Outcome 2: 0.2963
Outcome 3:

In [ ]:
for i in range(N_tr):
    U = qt.rand_unitary(d_out)

    # Extract the first 2 columns to form the isometry
    # We convert to a dense array to slice easily, then back to Qobj
    V_matrix = U.full()[:, :dim_in]

    # Wrap it in a Qobj and set the dimensions correctly
    V = qt.Qobj(V_matrix, dims=[dim_out, [dim_in]])

    rho_qt= qt.Qobj(rho_training[i], dims=[[2], [2]])

    rho_qt_evolved = (V * rho_qt * V.dag()).ptrace(1)
    

In [10]:
import pennylane as qml

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/3k/fd7stp7s1rsch3ck5w1hhcd80000gn/T/ipykernel_93001/2122156990.py", line 1, in <module>
    import pennylane as qml
  File "/opt/anaconda3/lib/python3.11/site-packages/pennylane/__init__.py", line 28, in <module>
    from pennylane import control_flow
  File "/opt/anaconda3/lib/python3.11/site-packages/pennylane/control_flow/__init__.py", line 24, in <module>
    from .for_loop import for_loop
  File "/opt/anaconda3/lib/python3.11/site-packages/pennylane/control_flow/for_loop.py", line 21, in <module>
    from pennylane.capture import FlatFn, enabled
ImportError: cannot import name 'FlatFn' from 'pennylane.capture' (/opt/anaconda3/lib/python3.11/site-packages/pennylane/capture/__init__.py)

During handling of the above exception, another exception occurred:

Traceback (